
# Step 1: Creating Tokens

In [2]:
pip install datasets

  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached pyarrow-25.0.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached multiprocess-0.70.19-py311-none-any.whl.metadata (7.5 kB)
Using cached datasets-5.0.1-py3-none-any.whl (559 kB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached multiprocess-0.70.19-py311-none-any.whl (144 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 3.4 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [datasets]3/4 [datasets]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install -U datasets


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [43]:
from datasets import load_dataset

ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

In [9]:
del ds
import gc
gc.collect()

1725

In [46]:
raw_text = "\n".join(ds["train"]["text"])

print("Total characters:", len(raw_text))
print(raw_text[:999])

Total characters: 540095682

 = Valkyria Chronicles III = 


 Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . 

 The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the gam

#### splitting sentences into different tokens 

In [4]:
import re 
text = "Hello, world! How's everything going today? I'm building a tokenizer."
result = re.split(r'(\s|[.,!?;:(){}\[\]<>\"\'`~@#$%^&*+=/\\|_-])', text)
result = [item.strip() for item in result if item.strip()] # removing wide spaces item.strip gets false for spaces
print(result)

['Hello', ',', 'world', '!', 'How', "'", 's', 'everything', 'going', 'today', '?', 'I', "'", 'm', 'building', 'a', 'tokenizer', '.']


In [47]:
preprocessed = re.split(r'(\s|[.,!?;:(){}\[\]<>\"\'`~@#$%^&*+=/\\|_-])', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])
print(len(preprocessed))

['=', 'Valkyria', 'Chronicles', 'III', '=', 'Senjō', 'no', 'Valkyria', '3', ':', 'Unrecorded', 'Chronicles', '(', 'Japanese', ':', '戦場のヴァルキュリア3', ',', 'lit', '.', 'Valkyria', 'of', 'the', 'Battlefield', '3', ')', ',', 'commonly', 'referred', 'to', 'as']
105067749


## Step 2: Creating Token IDs

In [48]:
word = sorted(set(preprocessed))
print(len(word))

608557


In [19]:
vocab = { token:integer for integer, token in enumerate(word) }
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('-', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [49]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encoder(self, text):
        preprocessed = re.split(
            r'(\s|[.,!?;:(){}\[\]<>\"\'`~@#$%^&*+=/\\|_-])',
            text
        )
        preprocessed = [x.strip() for x in preprocessed if x.strip()]
        return [self.str_to_int[x] for x in preprocessed]

    def decoder(self, ids):
        text = " ".join(self.int_to_str[i] for i in ids)
        return text


In [50]:
tokenizer = SimpleTokenizer(vocab)
text = """""It's the last he painted, you know,"
    Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encoder(text)
print(ids)
tokenizer.decoder(ids)

[1, 1, 56, 2, 858, 997, 605, 535, 750, 5, 1135, 599, 5, 1, 67, 7, 38, 859, 1117, 759, 800, 7]


'" " It \' s the last he painted , you know , " Mrs . Gisburn said with pardonable pride .'

# ADDING SPECIAL CONTEXT TOKENS

In [51]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [52]:
len(vocab.items())
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)


('𝕄', 608554)
('𝕡', 608555)
('🖕', 608556)
('<|endoftext|>', 608557)
('<|unk|>', 608558)


In [53]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encoder(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decoder(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [54]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "carrying over a large portion of the work"
text2 = "commonly referred to as Valkyria Chronicles III outside Japan"

text = " <|endoftext|> ".join((text1, text2))

print(text)

carrying over a large portion of the work <|endoftext|> commonly referred to as Valkyria Chronicles III outside Japan


In [55]:
tokenizer.encoder(text)


[437866,
 522862,
 415217,
 499124,
 533141,
 520083,
 569574,
 586694,
 608557,
 444706,
 542532,
 571373,
 423874,
 388042,
 85506,
 176782,
 522733,
 186865]

In [56]:
tokenizer.decoder(tokenizer.encoder(text))

'carrying over a large portion of the work <|endoftext|> commonly referred to as Valkyria Chronicles III outside Japan'